In [ ]:
import cv2
import numpy as np
import os

# Создание папки для сохранения кадров
output_folder = "saved_frames"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Загрузка модели YOLO (используем YOLOv4-tiny для скорости)
net = cv2.dnn.readNet("yolov4-tiny.weights", "yolov4-tiny.cfg")

# Получение имен всех слоев
layer_names = net.getLayerNames()

# Получение выходных слоев
output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers()]

# Загрузка классов
with open("coco.names", "r") as f:
    classes = [line.strip() for line in f.readlines()]

# Захват видеопотока
cap = cv2.VideoCapture("video_2025-01-19_21-33-09.mp4")

# Определение области интереса (ROI)
roi = [(130, 130), (600, 600)]  # Пример координат области

# Счетчик кадров
frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Уменьшение разрешения кадра
    frame = cv2.resize(frame, (640, 360))  # Уменьшение до 640x360

    # Пропуск кадров (обрабатываем каждый второй кадр)
    if frame_count % 2 != 0:
        frame_count += 1
        continue

    # Детекция объектов
    blob = cv2.dnn.blobFromImage(frame, 0.00392, (416, 416), (0, 0, 0), True, crop=False)
    net.setInput(blob)
    outs = net.forward(output_layers)

    # Обработка результатов детекции
    class_ids = []
    confidences = []
    boxes = []
    for out in outs:
        for detection in out:
            scores = detection[5:]
            class_id = np.argmax(scores)
            confidence = scores[class_id]
            if confidence > 0.5 and classes[class_id] == "car":  # Проверка класса "car"
                center_x = int(detection[0] * frame.shape[1])
                center_y = int(detection[1] * frame.shape[0])
                w = int(detection[2] * frame.shape[1])
                h = int(detection[3] * frame.shape[0])

                x = int(center_x - w / 2)
                y = int(center_y - h / 2)

                boxes.append([x, y, w, h])
                confidences.append(float(confidence))
                class_ids.append(class_id)

    # Применение Non-Maximum Suppression для удаления дубликатов
    indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)

    # Проверка попадания объекта в ROI и условия на площадь
    for i in range(len(boxes)):
        if i in indexes:
            x, y, w, h = boxes[i]

            # Вычисление площади bounding box машины
            bbox_area = w * h

            # Вычисление пересечения bounding box и ROI
            x1 = max(x, roi[0][0])
            y1 = max(y, roi[0][1])
            x2 = min(x + w, roi[1][0])
            y2 = min(y + h, roi[1][1])

            # Площадь пересечения
            intersection_area = max(0, x2 - x1) * max(0, y2 - y1)

            # Проверка, что минимум 0.1% bounding box находится в ROI
            if bbox_area > 0 and intersection_area / bbox_area >= 0.001:
                # Увеличение счетчика кадров
                frame_count += 1

                # Обрезка кадра по ROI
                roi_frame = frame[roi[0][1]:roi[1][1], roi[0][0]:roi[1][0]]

                # Сохранение обрезанного кадра
                output_path = os.path.join(output_folder, f"frame_{frame_count}.jpg")
                cv2.imwrite(output_path, roi_frame)
                print(f"Сохранен кадр: {output_path}")

    # Отображение кадра с выделенными объектами и ROI
    cv2.rectangle(frame, roi[0], roi[1], (0, 255, 0), 2)
    for i in range(len(boxes)):
        if i in indexes:
            x, y, w, h = boxes[i]
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)
    cv2.imshow("Frame", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()